<a href="https://colab.research.google.com" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 TrialForge — Synthetic Clinical Trial Data Engine
**Dataset: CDC Diabetes Health Indicators (BRFSS 2015) — 10,000 patients**


## Cell 1 — Install Dependencies

In [ ]:
!pip install -q sdv scikit-learn shap matplotlib seaborn scipy kagglehub


## Cell 2 — Load CDC Diabetes Health Indicators Dataset (10,000 patients) & Split 80/20


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
import kagglehub, os

# Download CDC Diabetes Health Indicators (BRFSS 2015)
path = kagglehub.dataset_download("alexteboul/diabetes-health-indicators-dataset")
csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
print(f"Available files: {csv_files}")

# Use the binary classification version
target_file = [f for f in csv_files if "binary" in f.lower() and "5050" not in f.lower()]
if target_file:
    csv_path = os.path.join(path, target_file[0])
else:
    csv_path = os.path.join(path, csv_files[0])
full_data = pd.read_csv(csv_path)
print(f"Full dataset: {full_data.shape}")

# Rename target column
if "Diabetes_binary" in full_data.columns:
    full_data = full_data.rename(columns={"Diabetes_binary": "target"})
elif "Diabetes_012" in full_data.columns:
    full_data = full_data.rename(columns={"Diabetes_012": "target"})
    full_data["target"] = (full_data["target"] > 0).astype(int)

# Clean column names (remove spaces)
full_data.columns = [c.strip().replace(" ", "_") for c in full_data.columns]
full_data["target"] = full_data["target"].astype(int)

# Stratified sample to 10,000 patients
data, _ = train_test_split(
    full_data, train_size=10000,
    random_state=42, stratify=full_data["target"]
)
data.reset_index(drop=True, inplace=True)

# 80/20 stratified split
train_real, test_real = train_test_split(
    data, test_size=0.2, stratify=data["target"], random_state=42
)

print(f"Sampled dataset: {data.shape[0]} patients, {data.shape[1]} columns")
print(f"Train set:       {len(train_real)} patients")
print(f"Test set:        {len(test_real)} patients (held out)")
print(f"Disease prevalence: {data['target'].mean():.1%}")
print()
data.head()


## Cell 3 — Train TVAE & Generate Synthetic Patients

TVAE (Tabular Variational Autoencoder) instead of a cGAN — more stable on small datasets, no mode collapse risk.

In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=train_real)

synthesizer = TVAESynthesizer(
    metadata,
    epochs=300,
    batch_size=32,
    compress_dims=(128, 64),
    decompress_dims=(64, 128),
    l2scale=1e-5,
    loss_factor=2
)
synthesizer.fit(train_real)

synthetic_data = synthesizer.sample(num_rows=len(train_real))

print("Synthetic patients generated!")
print(f"Shape: {synthetic_data.shape}")
print(f"Real disease prevalence:      {train_real['target'].mean():.1%}")
print(f"Synthetic disease prevalence:  {synthetic_data['target'].mean():.1%}")
print()
synthetic_data.head()

## Cell 4 — Per-Feature Distribution Check (KS Test + Wasserstein Distance)

**Mechanistic interpretability of the TVAE:** Did the TVAE learn each feature's distribution correctly? The KS test checks statistical similarity; Wasserstein distance quantifies how far apart the distributions are.


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, wasserstein_distance

features = [c for c in data.columns if c != "target"]

# Auto-detect continuous vs categorical based on unique values
continuous = [c for c in features if data[c].nunique() > 10]
categorical = [c for c in features if data[c].nunique() <= 10]

print(f"Continuous features ({len(continuous)}): {continuous}")
print(f"Categorical features ({len(categorical)}): {categorical}")
print()

print("=" * 70)
print(f"{'Feature':<20} {'KS Stat':>10} {'KS p-value':>12} {'Wasserstein':>12}  Verdict")
print("=" * 70)

ks_results = {}
for feat in features:
    ks_stat, ks_pval = ks_2samp(train_real[feat], synthetic_data[feat])
    wd = wasserstein_distance(train_real[feat], synthetic_data[feat])
    ks_results[feat] = {"ks_stat": ks_stat, "ks_pval": ks_pval, "wd": wd}
    mark = "GOOD" if ks_pval > 0.05 else "DIVERGENT ✗"
    print(f"{feat:<20} {ks_stat:>10.4f} {ks_pval:>12.4f} {wd:>12.4f}  {mark}")

# ── Continuous feature distributions ──
n_cont = len(continuous)
if n_cont > 0:
    n_rows_c = (n_cont + 4) // 5
    fig, axes = plt.subplots(n_rows_c, min(n_cont, 5), figsize=(22, 4 * n_rows_c))
    if n_cont == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)
    fig.suptitle("Real vs Synthetic — Continuous Features", fontsize=14, fontweight="bold")
    for i, feat in enumerate(continuous):
        r, c = i // 5, i % 5
        ax = axes[r][c] if n_rows_c > 1 or n_cont > 1 else axes[0][0]
        ax.hist(train_real[feat], bins=25, alpha=0.6, color="#2563EB", label="Real", density=True)
        ax.hist(synthetic_data[feat], bins=25, alpha=0.6, color="#D85A30", label="Synthetic", density=True)
        ax.set_title(f"{feat}\n(KS p={ks_results[feat]['ks_pval']:.3f})", fontsize=9)
        ax.legend(fontsize=7)
    # Hide unused axes
    for i in range(n_cont, n_rows_c * min(n_cont, 5)):
        r, c = i // 5, i % 5
        if r < axes.shape[0] and c < axes.shape[1]:
            axes[r][c].set_visible(False)
    plt.tight_layout()
    plt.show()

# ── Categorical feature distributions ──
n_cat = len(categorical)
if n_cat > 0:
    n_cols_plot = 4
    n_rows_cat = (n_cat + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows_cat, n_cols_plot, figsize=(20, 4 * n_rows_cat))
    axes = np.atleast_2d(axes)
    fig.suptitle("Real vs Synthetic — Categorical Features", fontsize=14, fontweight="bold")
    for idx, feat in enumerate(categorical):
        ax = axes[idx // n_cols_plot][idx % n_cols_plot]
        real_vc = train_real[feat].value_counts(normalize=True).sort_index()
        syn_vc = synthetic_data[feat].value_counts(normalize=True).sort_index()
        all_vals = sorted(set(real_vc.index) | set(syn_vc.index))
        x = np.arange(len(all_vals))
        w = 0.35
        ax.bar(x - w/2, [real_vc.get(v, 0) for v in all_vals], w, color="#2563EB", label="Real", alpha=0.8)
        ax.bar(x + w/2, [syn_vc.get(v, 0) for v in all_vals], w, color="#D85A30", label="Synthetic", alpha=0.8)
        ax.set_title(feat, fontsize=9)
        ax.set_xticks(x)
        ax.set_xticklabels([str(v) for v in all_vals], fontsize=7)
        ax.legend(fontsize=6)
    # Hide unused axes
    for idx in range(n_cat, n_rows_cat * n_cols_plot):
        axes[idx // n_cols_plot][idx % n_cols_plot].set_visible(False)
    plt.tight_layout()
    plt.show()


## Cell 5 — Cramér's V Correlation Matrix (Real vs Synthetic vs Difference)

**Interpretability angle:** Does the TVAE preserve pairwise feature associations? The difference heatmap reveals exactly which relationships it failed to learn.

In [ ]:
import seaborn as sns
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    """Bias-corrected Cramér's V."""
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    phi2 = chi2 / n
    r, k = ct.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1)) / (n-1))
    rcorr = r - ((r-1)**2) / (n-1)
    kcorr = k - ((k-1)**2) / (n-1)
    denom = min(kcorr - 1, rcorr - 1)
    return np.sqrt(phi2corr / denom) if denom > 0 else 0.0

def build_cramers_matrix(df, cols):
    n = len(cols)
    mat = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)
    for i in cols:
        for j in cols:
            mat.loc[i, j] = cramers_v(df[i], df[j])
    return mat

# Use top 12 features (by RF importance) + target to keep heatmap readable
from sklearn.ensemble import RandomForestClassifier as _RFC
_qrf = _RFC(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
_qrf.fit(train_real.drop("target", axis=1), train_real["target"])
_imp = pd.Series(_qrf.feature_importances_, index=features).sort_values(ascending=False)
top_feats = _imp.head(12).index.tolist()
cv_cols = top_feats + ["target"]
print(f"Using top {len(top_feats)} features for correlation matrices: {top_feats}")

# Build matrices
real_cv = build_cramers_matrix(train_real, cv_cols)
synth_cv = build_cramers_matrix(synthetic_data, cv_cols)
diff_cv = (real_cv - synth_cv).abs()

# Summary stats
mask_idx = np.triu_indices_from(diff_cv.values, k=1)
mean_diff = diff_cv.values[mask_idx].mean()
max_diff = diff_cv.values[mask_idx].max()

print(f"Cramér's V Fidelity Score (mean |diff|): {mean_diff:.4f}   (target: < 0.08)")
print(f"Max difference: {max_diff:.4f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(26, 8))

sns.heatmap(real_cv.astype(float), annot=True, fmt=".2f", cmap="Blues",
            ax=axes[0], vmin=0, vmax=1, annot_kws={"size": 7}, square=True)
axes[0].set_title("Real Data — Cramér's V", fontweight="bold", fontsize=12)
axes[0].tick_params(labelsize=7)

sns.heatmap(synth_cv.astype(float), annot=True, fmt=".2f", cmap="Blues",
            ax=axes[1], vmin=0, vmax=1, annot_kws={"size": 7}, square=True)
axes[1].set_title("Synthetic Data — Cramér's V", fontweight="bold", fontsize=12)
axes[1].tick_params(labelsize=7)

sns.heatmap(diff_cv.astype(float), annot=True, fmt=".2f", cmap="Reds",
            ax=axes[2], vmin=0, vmax=0.3, annot_kws={"size": 7}, square=True)
axes[2].set_title(f"Absolute Difference (mean={mean_diff:.3f})", fontweight="bold", fontsize=12)
axes[2].tick_params(labelsize=7)

plt.tight_layout()
plt.show()


## Cell 6 — Mutual Information Matrix (Real vs Synthetic)

MI captures non-linear dependencies that Cramér's V may miss — a deeper check on what the TVAE learned.

In [ ]:
from sklearn.metrics import mutual_info_score

def build_mi_matrix(df, cols):
    n = len(cols)
    mat = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)
    for i in cols:
        for j in cols:
            mat.loc[i, j] = mutual_info_score(df[i], df[j])
    return mat

real_mi = build_mi_matrix(train_real, cv_cols)
synth_mi = build_mi_matrix(synthetic_data, cv_cols)
diff_mi = (real_mi - synth_mi).abs()
mean_mi_diff = diff_mi.values[mask_idx].mean()

print(f"Mutual Information Fidelity Score (mean |diff|): {mean_mi_diff:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(26, 8))

sns.heatmap(real_mi.astype(float), annot=True, fmt=".2f", cmap="Greens",
            ax=axes[0], annot_kws={"size": 7}, square=True)
axes[0].set_title("Real Data — Mutual Information", fontweight="bold", fontsize=12)
axes[0].tick_params(labelsize=7)

sns.heatmap(synth_mi.astype(float), annot=True, fmt=".2f", cmap="Greens",
            ax=axes[1], annot_kws={"size": 7}, square=True)
axes[1].set_title("Synthetic Data — Mutual Information", fontweight="bold", fontsize=12)
axes[1].tick_params(labelsize=7)

sns.heatmap(diff_mi.astype(float), annot=True, fmt=".2f", cmap="Oranges",
            ax=axes[2], vmin=0, vmax=0.5, annot_kws={"size": 7}, square=True)
axes[2].set_title(f"MI Absolute Difference (mean={mean_mi_diff:.3f})", fontweight="bold", fontsize=12)
axes[2].tick_params(labelsize=7)

plt.tight_layout()
plt.show()


## Cell 7 — TSTR vs TRTR (Random Forest)

**TSTR** = Train on Synthetic, Test on Real. **TRTR** = Train on Real, Test on Real (baseline). The gap between them is how much information the TVAE lost.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix,
                             RocCurveDisplay)

X_train_real = train_real.drop("target", axis=1)
y_train_real = train_real["target"]
X_test_real = test_real.drop("target", axis=1)
y_test_real = test_real["target"]
X_synth = synthetic_data.drop("target", axis=1)
y_synth = synthetic_data["target"]

# ── TRTR: Train on Real, Test on Real (baseline) ──
rf_real = RandomForestClassifier(
    n_estimators=200, min_samples_split=5, min_samples_leaf=2,
    max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1
)
rf_real.fit(X_train_real, y_train_real)
pred_real = rf_real.predict(X_test_real)
proba_real = rf_real.predict_proba(X_test_real)[:, 1]

# ── TSTR: Train on Synthetic, Test on Real ──
rf_synth = RandomForestClassifier(
    n_estimators=200, min_samples_split=5, min_samples_leaf=2,
    max_features="sqrt", class_weight="balanced", random_state=42, n_jobs=-1
)
rf_synth.fit(X_synth, y_synth)
pred_synth = rf_synth.predict(X_test_real)
proba_synth = rf_synth.predict_proba(X_test_real)[:, 1]

# ── Compare ──
results = {
    "TRTR": {
        "accuracy": accuracy_score(y_test_real, pred_real),
        "f1": f1_score(y_test_real, pred_real),
        "auc_roc": roc_auc_score(y_test_real, proba_real),
    },
    "TSTR": {
        "accuracy": accuracy_score(y_test_real, pred_synth),
        "f1": f1_score(y_test_real, pred_synth),
        "auc_roc": roc_auc_score(y_test_real, proba_synth),
    },
}

print("=" * 60)
print(f"{'Metric':<12} {'TRTR (Real)':<16} {'TSTR (Synthetic)':<18} {'Gap':<10}")
print("=" * 60)
for m in ["accuracy", "f1", "auc_roc"]:
    trtr_val = results["TRTR"][m]
    tstr_val = results["TSTR"][m]
    gap = trtr_val - tstr_val
    ok = "✓" if abs(gap) < 0.05 else "✗"
    print(f"{m:<12} {trtr_val:<16.4f} {tstr_val:<18.4f} {gap:>+.4f}  {ok}")
print("=" * 60)

print("\n── TRTR Report ──")
print(classification_report(y_test_real, pred_real, target_names=["No Diabetes", "Diabetes"]))
print("── TSTR Report ──")
print(classification_report(y_test_real, pred_synth, target_names=["No Diabetes", "Diabetes"]))

# ── ROC Curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
RocCurveDisplay.from_estimator(rf_real, X_test_real, y_test_real, ax=axes[0], color="#2563EB")
axes[0].set_title(f"TRTR — AUC = {results['TRTR']['auc_roc']:.3f}", fontweight="bold")
RocCurveDisplay.from_estimator(rf_synth, X_test_real, y_test_real, ax=axes[1], color="#D85A30")
axes[1].set_title(f"TSTR — AUC = {results['TSTR']['auc_roc']:.3f}", fontweight="bold")
plt.tight_layout()
plt.show()

# ── Confusion Matrices ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, preds, title, cmap in [
    (axes[0], pred_real, "TRTR (Real-trained)", "Blues"),
    (axes[1], pred_synth, "TSTR (Synthetic-trained)", "Oranges"),
]:
    cm = confusion_matrix(y_test_real, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=["No Diabetes","Diabetes"],
                yticklabels=["No Diabetes","Diabetes"])
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## Cell 8 — SHAP Feature Importance: Real-Trained vs Synthetic-Trained

**Core interpretability finding.** If the SHAP rankings match between both models, the TVAE preserved the predictive structure. Where they diverge = what the TVAE failed to learn.

In [ ]:
import shap

# ── SHAP for real-trained model ──
explainer_real = shap.TreeExplainer(rf_real)
shap_vals_real = explainer_real.shap_values(X_test_real)

if isinstance(shap_vals_real, list):
    sv_real = shap_vals_real[1]
elif len(shap_vals_real.shape) == 3:
    sv_real = shap_vals_real[:, :, 1]
else:
    sv_real = shap_vals_real

# ── SHAP for synthetic-trained model ──
explainer_synth = shap.TreeExplainer(rf_synth)
shap_vals_synth = explainer_synth.shap_values(X_test_real)

if isinstance(shap_vals_synth, list):
    sv_synth = shap_vals_synth[1]
elif len(shap_vals_synth.shape) == 3:
    sv_synth = shap_vals_synth[:, :, 1]
else:
    sv_synth = shap_vals_synth

# ── Side-by-side beeswarm plots ──
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

plt.sca(axes[0])
shap.summary_plot(sv_real, X_test_real, plot_type="dot", show=False)
axes[0].set_title("SHAP — Real-Trained (TRTR)", fontweight="bold", fontsize=13)

plt.sca(axes[1])
shap.summary_plot(sv_synth, X_test_real, plot_type="dot", show=False)
axes[1].set_title("SHAP — Synthetic-Trained (TSTR)", fontweight="bold", fontsize=13)

plt.tight_layout()
plt.show()

# ── Rank comparison table ──
mean_real = pd.Series(np.abs(sv_real).mean(axis=0), index=X_test_real.columns).sort_values(ascending=False)
mean_synth = pd.Series(np.abs(sv_synth).mean(axis=0), index=X_test_real.columns).sort_values(ascending=False)

rank_r = mean_real.rank(ascending=False).astype(int)
rank_s = mean_synth.rank(ascending=False).astype(int)

print("=" * 65)
print(f"{'Feature':<12} {'|SHAP| Real':<14} {'|SHAP| Synth':<14} {'Real Rank':<10} {'Synth Rank':<11} Shift")
print("=" * 65)
for feat in mean_real.index:
    shift = int(rank_r[feat] - rank_s[feat])
    arrow = "—" if shift == 0 else f"↑{abs(shift)}" if shift > 0 else f"↓{abs(shift)}"
    flag = "  ← TVAE distorted" if abs(shift) >= 3 else ""
    print(f"{feat:<12} {mean_real[feat]:<14.4f} {mean_synth[feat]:<14.4f} {int(rank_r[feat]):<10} {int(rank_s[feat]):<11} {arrow}{flag}")
print("=" * 65)

# ── Bar chart comparison ──
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(mean_real))
w = 0.35
ax.barh(x - w/2, mean_real.values, w, color="#2563EB", label="Real-trained", alpha=0.85)
ax.barh(x + w/2, mean_synth.reindex(mean_real.index).values, w, color="#D85A30", label="Synthetic-trained", alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(mean_real.index)
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Feature Importance: Real-Trained vs Synthetic-Trained", fontweight="bold")
ax.legend(loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Cell 9 — SHAP Waterfall (Single Patient Explanation)

Per-patient breakdown: which features pushed this patient toward disease vs. no disease?

In [ ]:
patient_idx = 0

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plt.sca(axes[0])
ev_real = explainer_real.expected_value
base_real = ev_real[1] if isinstance(ev_real, (list, np.ndarray)) else ev_real
shap.plots.waterfall(
    shap.Explanation(
        values=sv_real[patient_idx],
        base_values=base_real,
        data=X_test_real.iloc[patient_idx],
        feature_names=list(X_test_real.columns)
    ), show=False
)
axes[0].set_title("Patient 0 — Real-Trained Model", fontweight="bold")

plt.sca(axes[1])
ev_synth = explainer_synth.expected_value
base_synth = ev_synth[1] if isinstance(ev_synth, (list, np.ndarray)) else ev_synth
shap.plots.waterfall(
    shap.Explanation(
        values=sv_synth[patient_idx],
        base_values=base_synth,
        data=X_test_real.iloc[patient_idx],
        feature_names=list(X_test_real.columns)
    ), show=False
)
axes[1].set_title("Patient 0 — Synthetic-Trained Model", fontweight="bold")

plt.tight_layout()
plt.show()

## Cell 10 — SHAP Dependence Plots (Top 4 Features)

Do the real-trained and synthetic-trained models agree on how each feature affects predictions? Differences here = TVAE failed to capture that interaction.

In [ ]:
top4 = mean_real.index[:4].tolist()

for feat in top4:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    plt.sca(axes[0])
    shap.dependence_plot(feat, sv_real, X_test_real, ax=axes[0], show=False)
    axes[0].set_title(f"{feat} — Real-Trained", fontweight="bold")

    plt.sca(axes[1])
    shap.dependence_plot(feat, sv_synth, X_test_real, ax=axes[1], show=False)
    axes[1].set_title(f"{feat} — Synthetic-Trained", fontweight="bold")

    plt.tight_layout()
    plt.show()

## Cell 11 — Summary Dashboard

All key metrics in one place.

In [ ]:
print("╔" + "═"*62 + "╗")
print("║        TRIALFORGE — CDC BRFSS 10K — RESULTS SUMMARY                       ║")
print("╠" + "═"*62 + "╣")
print(f"║  Cramér's V Fidelity (mean |diff|):  {mean_diff:<24.4f} ║")
print(f"║  Mutual Info Fidelity (mean |diff|):  {mean_mi_diff:<23.4f} ║")
print("╠" + "═"*62 + "╣")
print(f"║  TRTR Accuracy:  {results['TRTR']['accuracy']:<43.4f} ║")
print(f"║  TSTR Accuracy:  {results['TSTR']['accuracy']:<43.4f} ║")
print(f"║  Accuracy Gap:   {results['TRTR']['accuracy'] - results['TSTR']['accuracy']:>+.4f}{'':<38} ║")
print("╠" + "═"*62 + "╣")
print(f"║  TRTR AUC-ROC:   {results['TRTR']['auc_roc']:<43.4f} ║")
print(f"║  TSTR AUC-ROC:   {results['TSTR']['auc_roc']:<43.4f} ║")
print(f"║  AUC Gap:        {results['TRTR']['auc_roc'] - results['TSTR']['auc_roc']:>+.4f}{'':<38} ║")
print("╠" + "═"*62 + "╣")
print(f"║  TRTR F1:        {results['TRTR']['f1']:<43.4f} ║")
print(f"║  TSTR F1:        {results['TSTR']['f1']:<43.4f} ║")
print(f"║  F1 Gap:         {results['TRTR']['f1'] - results['TSTR']['f1']:>+.4f}{'':<38} ║")
print("╠" + "═"*62 + "╣")
print("║  SHAP Rank Shifts:                                          ║")
for feat in mean_real.index:
    shift = int(rank_r[feat] - rank_s[feat])
    arrow = "—" if shift == 0 else f"↑{abs(shift)}" if shift > 0 else f"↓{abs(shift)}"
    line = f"║    {feat:<10}  Real #{int(rank_r[feat]):<3} → Synth #{int(rank_s[feat]):<3}  ({arrow})"
    print(f"{line:<63} ║")
print("╚" + "═"*62 + "╝")

## Findings & Interpretability Write-up

### Research Question
When a TVAE generates synthetic diabetes patients, which clinical relationships does it faithfully learn, and which does it systematically fail to capture?

### Methodology
1. Trained TVAE on 80% of UCI Diabetes (242 patients, 14 features)
2. Generated 242 synthetic patients from the learned latent space
3. **Mechanistic interpretability of the TVAE:**
   - Per-feature KS tests → which marginal distributions the TVAE captured
   - Cramér's V matrices (real vs synthetic) → pairwise association fidelity
   - Mutual Information matrices → non-linear dependency fidelity
4. **Downstream validation:**
   - TSTR vs TRTR with Random Forest → information loss quantification
   - SHAP comparison → which predictive relationships the TVAE distorted

### Key Findings
*(Fill in after running — report the actual numbers)*

- **Cramér's V fidelity:** [mean_diff] (target < 0.08)
- **TSTR vs TRTR gap:** accuracy [X], F1 [X], AUC [X]
- **Features the TVAE learned well:** [list features with KS p > 0.05 AND rank shift ≤ 1]
- **Features the TVAE struggled with:** [list features with KS p < 0.05 OR rank shift ≥ 3]
- **Core insight:** "The TVAE faithfully reproduces [X, Y, Z] relationships but systematically [underestimates/distorts] the [A ↔ B] interaction, suggesting its latent space lacks capacity for [specific clinical relationship]."

### Regeneron Framing
Synthetic data can replace real patient data for ML training — but only after validation. Our pipeline gives researchers a quantitative trust certificate showing exactly where synthetic data is reliable and where it isn't.

### d_model Framing
We reverse-engineered a generative model by probing its outputs with statistical tests (Cramér's V, MI) and downstream behavioral analysis (TSTR + SHAP comparison). This is mechanistic interpretability applied to a VAE — understanding what it learned and where it fails.

### Future Work
- Increase TVAE latent dimensions to test if failed relationships can be recovered
- Conditional generation for demographic subgroup analysis
- Test on MIMIC-III for clinical-scale validation
- Deploy as FastAPI + React dashboard